In [ ]:
import os
import numpy as np
import pandas as pd
import pyabf
from datetime import datetime

In [ ]:
def analyze_holding_current_segmented(abf_file, time_window, sweep_window):
    """
    Analyzes holding current from a voltage clamp ABF file and returns the average 
    holding current before, during, and after an exclusion window.

    Parameters:
        abf_file (str): Path to ABF file.
        time_window (list): [start, end] in milliseconds for holding current calculation within each sweep.
        sweep_window (list): [start_sweep, end_sweep] for exclusion.

    Returns:
        dict: average holding current values for each segment.
    """
    abf = pyabf.ABF(abf_file)

    # Check if this is a voltage clamp recording
    if not any(unit in abf.sweepLabelY for unit in ["pA", "nA"]):
        raise ValueError(f"{abf_file} does not appear to be a voltage clamp recording.")

    start_time, end_time = time_window
    start_excl, end_excl = sweep_window

    before_current, during_current, after_current = [], [], []

    for sweep in range(abf.sweepCount):
        abf.setSweep(sweep)
        time_idx = (abf.sweepX * 1000 >= start_time) & (abf.sweepX * 1000 <= end_time)
        holding_current = np.mean(abf.sweepY[time_idx])

        if sweep < start_excl - 1:
            before_current.append(holding_current)
        elif start_excl - 1 <= sweep <= end_excl - 1:
            during_current.append(holding_current)
        else:
            after_current.append(holding_current)

    return {
        "before_avg_holding_current": np.mean(before_current) if before_current else np.nan,
        "during_avg_holding_current": np.mean(during_current) if during_current else np.nan,
        "after_avg_holding_current": np.mean(after_current) if after_current else np.nan
    }

In [ ]:
# === Customize these ===
data_directory = './data/05192025/'  # e.g., "/Users/you/data/"
time_window = [0, 500]                # in milliseconds
sweep_window = [15, 25]                # e.g., exclude sweeps 6 through 11
protocol_filter = "3_CC_spont"
save_output_csv = True
csv_filename = "rmp_summary.csv"

# === Run analysis ===
rmp_summary_df = summarize_rmp_across_files(
    data_directory,
    time_window,
    sweep_window,
    protocol_name=protocol_filter,
    save_csv=save_output_csv,
    csv_name=csv_filename
)

rmp_summary_df